# Load and process Klebsiella pneumoniae data

In [1]:
import numpy as np
import pandas as pd
from Bio import AlignIO, Phylo
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from Bio.Align import MultipleSeqAlignment
from tqdm import tqdm
import sys
sys.path.append('../pysimARG')
from segment_summary_stats import segment_summary_stats
from clonal_genealogy import ClonalTree

## Load and check `.aln` data

In [2]:
def remove_gap_columns(alignment_block):
    """
    Takes a Biopython MultipleSeqAlignment block and returns a new block
    with any column containing '-', '.', 'N', or 'n' completely removed,
    along with the list of valid column indices within this block.
    """
    align_len = alignment_block.get_alignment_length()

    invalid_chars = {'N', 'n', '-', '.'}
    
    valid_columns = [
        i for i in range(align_len) 
        if not any(char in alignment_block[:, i] for char in invalid_chars)
    ]

    cleaned_records = []
    for record in alignment_block:
        clean_seq_string = "".join([record.seq[i] for i in valid_columns])
        new_record = SeqRecord(Seq(clean_seq_string), id=record.id, description="")
        cleaned_records.append(new_record)

    # Now returning a tuple of (cleaned_alignment, valid_indices)
    return MultipleSeqAlignment(cleaned_records), valid_columns

In [3]:
def process_alignment_in_blocks(full_alignment, block_size=10000):
    """
    Slices a massive alignment into smaller blocks, cleans them using 
    remove_gap_columns(), and glues them back together. 
    Returns the cleaned alignment and a list mapping clean_index -> raw_index.
    """
    total_len = full_alignment.get_alignment_length()
    final_alignment = None
    
    # Map: clean_index -> raw_index
    position_mapping = []
    
    for start in tqdm(range(0, total_len, block_size), desc="processing alignment blocks"):
        end = min(start + block_size, total_len)
        sub_block = full_alignment[:, start:end]
        cleaned_block, valid_cols = remove_gap_columns(sub_block)

        if cleaned_block.get_alignment_length() > 0:
            if final_alignment is None:
                final_alignment = cleaned_block
            else:
                final_alignment += cleaned_block
            
            # Shift the valid block indices by the 'start' offset of the current block
            # and append them to global mapping.
            position_mapping.extend([col + start for col in valid_cols])
                
    return final_alignment, position_mapping

In [4]:
seq_path = "../data/klebsiella/noref.aln"
raw_alignment = AlignIO.read(seq_path, "fasta")

print("Alignment summary:")
print(raw_alignment)
print("-" * 40)

seq_count = 0
seq_names = []

for record in raw_alignment:
    print(record.id, len(record.seq))
    seq_names.append(record.id)
    seq_count += 1

print(f"Total number of sequences: {seq_count}")

Alignment summary:
Alignment with 100 rows and 5280085 columns
GTGTCACTTTCGCTTTGGCAGCAGTGTCTTGCCCGATTGCAGGA...GCC 19UUUUQvAAyHsrWsprUbF7-GCA_019927745
GTGTCACTTTCGCTTTGGCAGCAGTGTCTTGCCCGATTGCAGGA...GCC 1CCAAZiMef1Wi3JBZomcfi-GCA_019928075
GTGTCACTTTCGCTTTGGCAGCAGTGTCTTGCCCGATTGCAGGA...GCC 1J25vPYcr5cWPUCJFGcaYk-GCA_019928505
GTGTCACTTTCGCTTTGGCAGCAGTGTCTTGCCCGATTGCAGGA...GCC 1NBFH6Ut6cUUtx6LSWophX-GCA_019927505
GTGTCACTTTCGCTTTGGCAGCAGTGTCTTGCCCGATTGCAGGA...GCC 1jxgrGNJjS7terz8orucSc-GCA_019927405
GTGTCACTTTCGCTTTGGCAGCAGTGTCTTGCCCGATTGCAGGA...GCC 1k2mmojDYi7HwWwByEje5E-GCA_019928115
GTGTCACTTTCGCTTTGGCAGCAGTGTCTTGCCCGATTGCAGGA...GCC 2vb3nFYf4FyUm6srAtv2QZ-GCA_019930305
GTGTCACTTTCGCTTTGGCAGCAGTGTCTTGCCCGATTGCAGGA...GCC 35JJmVwexb5WeHSxNcpWqb-GCA_020407065
GTGTCACTTTCGCTTTGGCAGCAGTGTCTTGCCCGATTGCAGGA...GCC 3JR2tMy7o1jC2oFuU8nHzv-GCA_019930175
GTGTCACTTTCGCTTTGGCAGCAGTGTCTTGCCCGATTGCAGGA...GCC 3Z26tDKznNidwX8jD2CCXz-GCA_019927605
GTGTCACTTTCGCTTTGGCAGCAGTGTCTTGCCCGATTGCAGGA...GCC 3mN8Tu

In [5]:
alignment, clean_to_raw_map = process_alignment_in_blocks(raw_alignment, block_size=50000)

print("Alignment summary:")
print(alignment)
print("-" * 40)

seq_count = 0
seq_names = []

for record in alignment:
    print(record.id, len(record.seq))
    seq_names.append(record.id)
    seq_count += 1

print(f"Total number of sequences: {seq_count}")

processing alignment blocks: 100%|██████████| 106/106 [27:17<00:00, 15.44s/it]

Alignment summary:
Alignment with 100 rows and 3761346 columns
GCGATAACACGCTGGCACTGTATGCGCCAAACCGTTTTGTGCTC...CAG 19UUUUQvAAyHsrWsprUbF7-GCA_019927745
GCGATAACACGCTGGCACTGTATGCGCCAAACCGTTTTGTGCTC...CAG 1CCAAZiMef1Wi3JBZomcfi-GCA_019928075
GCGATAACACGCTGGCACTGTATGCGCCAAACCGTTTTGTGCTC...CAG 1J25vPYcr5cWPUCJFGcaYk-GCA_019928505
GCGATAACACGCTGGCACTGTATGCGCCAAACCGTTTTGTGCTC...CAG 1NBFH6Ut6cUUtx6LSWophX-GCA_019927505
GCGATAACACGCTGGCACTGTATGCGCCAAACCGTTTTGTGCTC...CAG 1jxgrGNJjS7terz8orucSc-GCA_019927405
GCGATAACACGCTGGCACTGTATGCGCCAAACCGTTTTGTGCTC...CAG 1k2mmojDYi7HwWwByEje5E-GCA_019928115
GCGATAACACGCTGGCACTGTATGCGCCAAACCGTTTTGTGCTC...CAG 2vb3nFYf4FyUm6srAtv2QZ-GCA_019930305
GCGATAACACGCTGGCACTGTATGCGCCAAACCGTTTTGTGCTC...CAG 35JJmVwexb5WeHSxNcpWqb-GCA_020407065
GCGATAACACGCTGGCACTGTATGCGCCAAACCGTTTTGTGCTC...CAG 3JR2tMy7o1jC2oFuU8nHzv-GCA_019930175
GCGATAACACGCTGGCACTGTATGCGCCAAACCGTTTTGTGCTC...CAG 3Z26tDKznNidwX8jD2CCXz-GCA_019927605
GCGATAACACGCTGGCACTGTATGCGCCAAACCGTTTTGTGCTC...CAG 3mN8Tu

## Trace back to raw sequence positions

### `rand_seg_df`

In [13]:
rand_seg_df = pd.read_csv("../data/klebsiella/rand_seg_info.csv")
rand_seg_df['Gene_Length_raw'] = None
rand_seg_df['Start_pos_raw'] = None
rand_seg_df['End_pos_raw'] = None
rand_seg_df.head()

,Gene_Length,Start_pos,End_pos,Alignment,Gene_Length_raw,Start_pos_raw,End_pos_raw
0,1000,1873417,1874416,True,None,None,None
1,1000,1104665,1105664,True,None,None,None
2,1000,210756,211755,True,None,None,None
3,1000,2961240,2962239,True,None,None,None
4,1000,65616,66615,True,None,None,None


In [14]:
for i in range(rand_seg_df.shape[0]):
    start_pos_clean = rand_seg_df.loc[i, 'Start_pos']
    end_pos_clean = rand_seg_df.loc[i, 'End_pos']
    
    rand_seg_df.loc[i, 'Start_pos_raw'] = clean_to_raw_map[start_pos_clean - 1] + 1
    rand_seg_df.loc[i, 'End_pos_raw'] = clean_to_raw_map[end_pos_clean - 1] + 1
    rand_seg_df.loc[i, 'Gene_Length_raw'] = rand_seg_df.loc[i, 'End_pos_raw'] - rand_seg_df.loc[i, 'Start_pos_raw'] + 1

print(rand_seg_df.head())
rand_seg_df.to_csv("../data/klebsiella/rand_seg_info_updated.csv", index=False)

   Gene_Length  Start_pos  End_pos  Alignment Gene_Length_raw Start_pos_raw  \
0         1000    1873417  1874416       True            1000       2407524   
1         1000    1104665  1105664       True            1000       1372809   
2         1000     210756   211755       True            1000        269646   
3         1000    2961240  2962239       True            1000       4233501   
4         1000      65616    66615       True            1001         69950   

  End_pos_raw  
0     2408523  
1     1373808  
2      270645  
3     4234500  
4       70950  


### `rand_seg_df2`

In [16]:
rand_seg_df2 = pd.read_csv("../data/klebsiella/rand_seg_info2.csv")
rand_seg_df2['Gene_Length_raw'] = None
rand_seg_df2['Start_pos_raw'] = None
rand_seg_df2['End_pos_raw'] = None
rand_seg_df2.head()

,Gene_Length,Start_pos,End_pos,Alignment,Gene_Length_raw,Start_pos_raw,End_pos_raw
0,7036,1873417,1880452,True,None,None,None
1,8139,210756,218894,True,None,None,None
2,179,2961240,2961418,True,None,None,None
3,450,212913,213362,True,None,None,None
4,4249,1980341,1984589,True,None,None,None


In [17]:
for i in range(rand_seg_df2.shape[0]):
    start_pos_clean = rand_seg_df2.loc[i, 'Start_pos']
    end_pos_clean = rand_seg_df2.loc[i, 'End_pos']
    
    rand_seg_df2.loc[i, 'Start_pos_raw'] = clean_to_raw_map[start_pos_clean - 1] + 1
    rand_seg_df2.loc[i, 'End_pos_raw'] = clean_to_raw_map[end_pos_clean - 1] + 1
    rand_seg_df2.loc[i, 'Gene_Length_raw'] = rand_seg_df2.loc[i, 'End_pos_raw'] - rand_seg_df2.loc[i, 'Start_pos_raw'] + 1

print(rand_seg_df2.head())
rand_seg_df2.to_csv("../data/klebsiella/rand_seg_info2_updated.csv", index=False)

   Gene_Length  Start_pos  End_pos  Alignment Gene_Length_raw Start_pos_raw  \
0         7036    1873417  1880452       True            8436       2407524   
1         8139     210756   218894       True            8772        269646   
2          179    2961240  2961418       True             179       4233501   
3          450     212913   213362       True             451        272414   
4         4249    1980341  1984589       True            4251       2739453   

  End_pos_raw  
0     2415959  
1      278417  
2     4233679  
3      272864  
4     2743703  
